In [1]:
# ----- 1. using numpy -----

In [1]:
import numpy as np

In [3]:
w1 = np.array([[.2, -.5, .1, 2.],
              [1.5, 1.3, 2.1, 0.],
              [0., .25, .2, -.3]])
data = np.array([56, 231, 24, 2]).reshape(4, 1)
b1 = np.array([1.1, 3.2, -1.2]).reshape(3, 1)

w2 = np.array([.3, 1.5, -.5]).reshape(1, 3)
b2 = 0.8

p = 0.5

In [4]:
def train_step(data, p):
    # p = dropout rate
    h1 = np.maximum(0, np.dot(w1, data) + b1) 
    print(h1, 'h1')
    u1 = (np.random.rand(*h1.shape) < p)  # p보다 작은 원소를 false로 만들어줌
    print(u1, 'u1')
    h1 *= u1 # True이면 1을 곱하고 False이먄 0을 곱함으로써 dropout
    print(h1, 'h1')
    h2 = np.maximum(0, np.dot(w2, h1) + b2) # h1의 값들 중 일부가 사라지므로 h2의 값이 전반적으로 줄어들 것임
    print(h2, 'h2')
    u2 = (np.random.rand(*h2.shape) < p)
    print(u2, 'u2')
    h2 *= u2
    print(h2, 'h2')

def test_step(data, p):
    h1 = np.maximum(0, np.dot(w1, data) + b1) * p # p를 곱함으로써 줄어든 값을 맞춰줌. 그러나 test가 p값에 영향을 받는 건 바람직하지 않음
    print('h1', h1)
    h2 = np.maximum(0, np.dot(w2, h1) + b2) * p 
    print('h2', h2)   

In [5]:
train_step(data, p)

[[  0.  ]
 [437.9 ]
 [ 60.75]] h1
[[ True]
 [False]
 [ True]] u1
[[ 0.  ]
 [ 0.  ]
 [60.75]] h1
[[0.]] h2
[[ True]] u2
[[0.]] h2


In [6]:
test_step(data, p)

h1 [[  0.   ]
 [218.95 ]
 [ 30.375]]
h2 [[157.01875]]


In [7]:
def train_step(data, p):
    # p = dropout rate
    h1 = np.maximum(0, np.dot(w1, data) + b1) 
    print(h1, 'h1')
    u1 = (np.random.rand(*h1.shape) < p) / p # h1의 값들 중 일부가 사라지므로 h2의 값이 전반적으로 줄어들텐데 이를 방지하기 위해 p로 나눠사 깂을 늘림
    print(u1, 'u1')
    h1 *= u1 
    print(h1, 'h1')
    h2 = np.maximum(0, np.dot(w2, h1) + b2)
    print(h2, 'h2')
    u2 = (np.random.rand(*h2.shape) < p) / p
    print(u2, 'u2')
    h2 *= u2
    print(h2, 'h2')

def test_step(data):
    h1 = np.maximum(0, np.dot(w1, data) + b1) # test에서는 p와 무관하게 작동하도록 함
    print('h1', h1)
    h2 = np.maximum(0, np.dot(w2, h1) + b2) 
    print('h2', h2)

In [10]:
train_step(data, p)

[[  0.  ]
 [437.9 ]
 [ 60.75]] h1
[[0.]
 [2.]
 [0.]] u1
[[  0. ]
 [875.8]
 [  0. ]] h1
[[1314.5]] h2
[[2.]] u2
[[2629.]] h2


In [11]:
# ----- 2. pytorch -----

In [2]:
import torch
import torch.nn as nn

In [3]:
class Dropout(nn.Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()
        if p < 0 or p > 1:
            raise ValueError("dropout probability has to be between 0 and 1, "
                             "but got {}".format(p))
        self.p = p

    def forward(self, input):
        mask = (np.random.rand(*input.shape) < self.p) / self.p
        output = input * torch.Tensor(mask)
        return output

    def __repr__(self):
        return self.__class__.__name__ + '(' + 'p=' + str(self.p) + ')'

In [4]:
x = torch.rand((3, 2))
x

tensor([[0.6052, 0.8495],
        [0.5471, 0.1660],
        [0.9825, 0.5723]])

In [5]:
Dropout(p=0.5)(x)

tensor([[0.0000, 0.0000],
        [0.0000, 0.3319],
        [0.0000, 1.1446]])